In [3]:

import os 
from dotenv import load_dotenv , find_dotenv
_ = load_dotenv(find_dotenv())  # read local .env file

openai_api_key = os.environ.get("OPENAI_API_KEY")

# Chat model
from langchain_openai import ChatOpenAI
llmModel = ChatOpenAI(model="gpt-4.1-mini")


In [ ]:
# chat completion model 


"""
In a chat prompt, each message has a role:

system sets the assistant’s behavior or context, such as “You are a historian.”
human contains the user’s input or question.
ai represents something the assistant said earlier in the conversation.
In your template, ("ai", "Sure!") is treated as a previous assistant reply. It’s optional; include it only when you want the model to see that earlier turn as conversation context.
"""


from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate  

# so we can create a chat prompt template with system and human messages.


chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "you are an {profession} expert on {topic}."),
        ("human", "Tell me one curious thing about {topic}."),
        ("ai" , "Sure !") ,
        ("human" , "{user_input}")
    ]
)


messages = chat_template.format_messages(
    profession="historian",
    topic="Kennedy family",
    user_input="Tell me one curious thing about JFK."
)
response = llmModel.invoke(messages)


print(response)






content='Certainly! A curious fact about John F. Kennedy is that he was the first U.S. president to win a Pulitzer Prize. He received it in 1957 for his book *Profiles in Courage*, which highlighted acts of bravery and integrity by American senators. Interestingly, there has been some debate over how much of the writing Kennedy personally did, with some suggesting that his speechwriter, Ted Sorensen, played a major role in drafting the book. Nonetheless, it remains a unique accolade in presidential history.' response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 47, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_557a6d3485', 'finish_reason': 'stop', 'logprobs': None} id='run-12e95e25-91fe-4eb1-9860-ad22ac35d3aa-0' usage_

## Few-Shot Prompting

Few-shot prompting gives a model a small number of examples in the prompt before the new question. The examples show the kind of answer or format you want, helping the model follow the pattern. They do not retrain the model; they are included as context in that request.

In [6]:
few_shot_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a historian specializing in the Kennedy family. Answer with one concise, specific fact."),
        ("human", "Who was JFK's younger brother?"),
        ("ai", "Robert F. Kennedy was JFK's younger brother. He later served as U.S. attorney general and a U.S. senator."),
        ("human", "What did Jacqueline Kennedy do as first lady?"),
        ("ai", "She led a restoration of the White House and helped bring attention to its history and collections."),
        ("human", "{question}"),
    ]
)

few_shot_messages = few_shot_template.format_messages(
    question="Tell me one curious thing about JFK."
)
few_shot_response = llmModel.invoke(few_shot_messages)
print(few_shot_response.content)

JFK won a Pulitzer Prize in 1957 for his book "Profiles in Courage."


# Chain

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "hi!", "output": "hola!"},
    {"input": "bye!", "output": "adios!"},
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates English to Spanish."),
        few_shot_prompt,
        ("human", "{user_input}"),
    ]
)
"""
    means “format the input with final_prompt, then pass the resulting messages to 
    llmModel.” Calling chain.invoke({"user_input": "How are you?"}) runs 
    both steps and returns the model’s response.

    It’s LangChain’s composition operator here, not a shell pipe.
"""

# what is this pipe symbo?
# 
chain = final_prompt | llmModel
chain_response = chain.invoke({"user_input": "How are you?"})



print(chain_response.content)

¿Cómo estás?


# LangChain Expression Language (LCEL)

LCEL is a declarative way to combine LangChain components, called **runnables**, into a workflow. The pipe operator (`|`) connects each step: the output of one step becomes the input of the next.

A common pattern is `prompt | model | output_parser`:

- The prompt turns input values into chat messages.
- The model generates a response.
- The output parser converts the response into a convenient format, such as plain text.

Use `invoke()` for one input, `batch()` for multiple inputs, and `stream()` to receive output in chunks. These methods execute the model and may make API requests.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# from messages is a method that creates a prompt template from a list of messages, where each message is a tuple of (role, content). The role can be "system", "human", or "ai", and the content can include placeholders for variables that will be filled in later. This allows you to create a structured conversation with the model, where you can specify the behavior of the assistant, the user's input, and any previous responses from the assistant.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a concise historian. Give one interesting, factual detail about the topic."),
        ("human", "Tell me one interesting fact about {topic}."),
    ]
)

# Compose prompt formatting, model generation, and conversion to plain text.
lcel_chain = prompt | llmModel | StrOutputParser()

# Run once with a single input.
answer = lcel_chain.invoke({"topic": "JFK"})
print(answer)

# Optional: uncomment these examples to make additional model requests.
# answers = lcel_chain.batch([
#     {"topic": "JFK"},
#     {"topic": "Jackie Kennedy"},
# ])
# print(answers)

# for chunk in lcel_chain.stream({"topic": "JFK"}):
#     print(chunk, end="", flush=True)

John F. Kennedy was the first U.S. president to hold a live televised press conference, beginning in 1961.


# Output Parsers

An output parser converts a model's response into a useful Python value. Chat models usually return an `AIMessage`; putting a parser after the model in an LCEL chain transforms that result.

- `StrOutputParser` extracts plain text. It is useful when you want a string instead of an `AIMessage` (see the LCEL example above).
- `JsonOutputParser` parses JSON text into Python dictionaries or lists.
- `PydanticOutputParser` parses the response into a Pydantic model and validates its fields and types.

For structured output, include the parser's format instructions in the prompt. Parsing and schema validation help with structure, but you should still handle invalid or unexpected model output.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field

"""
    Parse JSON: JsonOutputParser.parse(json_text) converts the JSON string into a 
    Python dictionary, so you can access values like json_result["year"].
    Validate with Pydantic: HistoricalFact defines the required 
    subject and fact fields. PydanticOutputParser.parse(example_json) 
    parses the JSON and validates it, returning a HistoricalFact object. 
    You can then use parsed_fact.fact.
    Build an LCEL chain: structured_prompt includes formatting instructions 
    generated from the Pydantic schema. The chain passes input through the prompt, 
    llmModel, then fact_parser, which returns a validated HistoricalFact.

"""



# JsonOutputParser converts JSON text into ordinary Python values.
json_parser = JsonOutputParser()
json_text = '{"person": "John F. Kennedy", "year": 1961}'
json_result = json_parser.parse(json_text)
print(json_result)


# PydanticOutputParser also validates the expected fields and types.
class HistoricalFact(BaseModel):
    subject: str = Field(description="Person or topic the fact is about")
    fact: str = Field(description="One concise historical fact")


fact_parser = PydanticOutputParser(pydantic_object=HistoricalFact)
example_json = (
    '{"subject": "John F. Kennedy", '
    '"fact": "He was the first U.S. president born in the 20th century."}'
)
parsed_fact = fact_parser.parse(example_json)
print(parsed_fact)
print(parsed_fact.fact)


# Compose a structured-output chain. Invoke it to make a model API request.
structured_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer with one factual historical detail.\n{format_instructions}"),
        ("human", "Give one interesting fact about {topic}."),
    ]
).partial(format_instructions=fact_parser.get_format_instructions())

structured_chain = structured_prompt | llmModel | fact_parser
parsed_response = structured_chain.invoke({"topic": "JFK"})
print(parsed_response.fact)

{'person': 'John F. Kennedy', 'year': 1961}
subject='John F. Kennedy' fact='He was the first U.S. president born in the 20th century.'
He was the first U.S. president born in the 20th century.
John F. Kennedy was the first U.S. president born in the 20th century.


# Get Format Instructions

In [ ]:
class Joke(BaseModel):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")


# JsonOutputParser adds schema-based format instructions and parses JSON into a dictionary.
# Use PydanticOutputParser when you also need the result validated as a Joke object.
parser = JsonOutputParser(pydantic_object=Joke)

# what is parser.get_format_instructions()?
# it returns a string with instructions for the model to output 
# JSON that matches the Joke schema. The instructions specify the 
# required fields (setup and punchline) and their types, 
# so the model knows how to format its response correctly. 
# This ensures that when the model generates a joke, 
# it will be structured in a way that can be parsed into a Joke object.


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write one short, family-friendly joke about {topic}.\n{format_instructions}",
        ),
        ("human", "Create a joke about {topic}."),
    ]
).partial(format_instructions=parser.get_format_instructions())
# what is partial method?
# The partial method is used to create a new prompt template with some of the 
# variables pre-filled
# in the above example, the format_instructions variable is pre-filled with 
# the output of parser.get_format_instructions(). This means that when you invoke the prompt, 
# you only need to provide the topic variable, and the format_instructions 
# will already be included in the prompt. This allows for more convenient and reusable prompt 
# templates.



joke_chain = prompt | llmModel | parser
joke_result = joke_chain.invoke({"topic": "programmers"})

print(joke_result)
print("Setup:", joke_result["setup"])
print("Punchline:", joke_result["punchline"])




{'setup': 'Why do programmers always mix up Christmas and Halloween?', 'punchline': 'Because Oct 31 == Dec 25!'}
Setup: Why do programmers always mix up Christmas and Halloween?
Punchline: Because Oct 31 == Dec 25!
